In [3]:
import pandas as pd

df = pd.read_csv("Source 1 CLEANED.csv")
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df_clean = df[(df["duration_seconds"].notna()) & (df["duration_seconds"] != 0)].copy()
df_clean.to_csv("Source-1-NO_ZERO_DURATION.csv", index=False)

print(len(df), len(df_clean), len(df) - len(df_clean))


1742 1738 4


In [4]:
df

,url,title,category,duration_seconds,view_count,like_count,comment_count
0,https://www.youtube.com/watch?v=_uQrJ0TkZlc,Python Full Course for Beginners,python tutorial,22447,46484976,1226186,61315
1,https://www.youtube.com/watch?v=kqtD5dpn9C8,Python for Beginners - Learn Coding with Pytho...,python tutorial,3606,23266146,532774,26625
2,https://www.youtube.com/watch?v=rfscVS0vtbw,Learn Python - Full Course for Beginners [Tuto...,python tutorial,16012,48131277,1109421,45939
3,https://www.youtube.com/watch?v=t8pPdKYpowI,Python Tutorial for Beginners - Learn Python i...,python tutorial,19890,6599656,167519,4008
4,https://www.youtube.com/watch?v=K5KVEU3aaeQ,Python Full Course for Beginners [2025],python tutorial,7341,4613318,112457,3020
...,...,...,...,...,...,...,...
1737,https://www.youtube.com/watch?v=VjzsAJa27mU,"Zhao Yunshan, Chongqing April 12, 2024 #chinat...",travelling,14,19465999,432499,10296
1738,https://www.youtube.com/watch?v=4jrk81foQx8,"PACK, PREP AND TRAVEL W ME TO HAWAII FOR A MON...",travelling,1014,111869,3228,80
1739,https://www.youtube.com/watch?v=e7pkte-fuOU,Wonders of Greece | Most Amazing Places in Gre...,travelling,3607,439933,3143,132
1740,https://www.youtube.com/watch?v=x_ywByCJ0hk,Things to BEWARE OF IN SOUTH KOREA 😬 #travel #...,travelling,60,368443,5535,594


In [5]:
import pandas as pd

df = pd.read_csv("Source 2 CLEANED.csv")
df["duration_seconds"] = pd.to_numeric(df["duration_seconds"], errors="coerce")
df_clean = df[(df["duration_seconds"].notna()) & (df["duration_seconds"] != 0)].copy()
df_clean.to_csv("Source-2-NO_ZERO_DURATION.csv", index=False)

print(len(df), len(df_clean), len(df) - len(df_clean))


451 451 0


In [9]:
import pandas as pd

bins = [0, 60, 180, 600, float("inf")]
labels = ["<=60", "61-180", "181-600", ">600"]

# ========= Source 1 =========
s1 = pd.read_csv("Source-1-NO_ZERO_DURATION.csv")
s1 = s1[s1["duration_seconds"] != 0].copy()

s1["duration_bucket"] = pd.cut(s1["duration_seconds"], bins=bins, labels=labels)

target_n_1 = 600

counts1 = s1["duration_bucket"].value_counts().sort_index()
weights1 = counts1 / counts1.sum()
alloc1 = (weights1 * target_n_1).round().astype(int)

diff1 = target_n_1 - alloc1.sum()
if diff1 != 0:
    order1 = (weights1 * target_n_1 - (weights1 * target_n_1).round()).abs().sort_values().index
    alloc1.loc[order1[:abs(diff1)]] += 1 if diff1 > 0 else -1

s1_strat = (
    s1.groupby("duration_bucket", group_keys=False)
      .apply(lambda x: x.sample(n=min(len(x), alloc1.loc[x.name]), random_state=42))
      .reset_index(drop=True)
)

s1_strat = s1_strat.sample(n=min(target_n_1, len(s1_strat)), random_state=42).reset_index(drop=True)
s1_strat.to_csv("Source-1-STRATIFIED_600.csv", index=False)

print("SOURCE 1")
print("Original bucket counts:")
print(counts1)
print("\nAllocated sample sizes:")
print(alloc1)
print("\nFinal bucket counts:")
print(s1_strat["duration_bucket"].value_counts().sort_index())
print("Total rows:", len(s1_strat))
print("-" * 40)

# ========= Source 2 =========
s2 = pd.read_csv("Source-2-NO_ZERO_DURATION.csv")
s2 = s2[s2["duration_seconds"] != 0].copy()

s2["duration_bucket"] = pd.cut(s2["duration_seconds"], bins=bins, labels=labels)

target_n_2 = 400

counts2 = s2["duration_bucket"].value_counts().sort_index()
weights2 = counts2 / counts2.sum()
alloc2 = (weights2 * target_n_2).round().astype(int)

diff2 = target_n_2 - alloc2.sum()
if diff2 != 0:
    order2 = (weights2 * target_n_2 - (weights2 * target_n_2).round()).abs().sort_values().index
    alloc2.loc[order2[:abs(diff2)]] += 1 if diff2 > 0 else -1

s2_strat = (
    s2.groupby("duration_bucket", group_keys=False)
      .apply(lambda x: x.sample(n=min(len(x), alloc2.loc[x.name]), random_state=42))
      .reset_index(drop=True)
)

s2_strat = s2_strat.sample(n=min(target_n_2, len(s2_strat)), random_state=42).reset_index(drop=True)
s2_strat.to_csv("Source-2-STRATIFIED_400.csv", index=False)

print("SOURCE 2")
print("Original bucket counts:")
print(counts2)
print("\nAllocated sample sizes:")
print(alloc2)
print("\nFinal bucket counts:")
print(s2_strat["duration_bucket"].value_counts().sort_index())
print("Total rows:", len(s2_strat))


SOURCE 1
Original bucket counts:
duration_bucket
<=60       1125
61-180      103
181-600     124
>600        386
Name: count, dtype: int64

Allocated sample sizes:
duration_bucket
<=60       388
61-180      36
181-600     43
>600       133
Name: count, dtype: int64

Final bucket counts:
duration_bucket
<=60       388
61-180      36
181-600     43
>600       133
Name: count, dtype: int64
Total rows: 600
----------------------------------------
SOURCE 2
Original bucket counts:
duration_bucket
<=60       164
61-180      16
181-600    156
>600       115
Name: count, dtype: int64

Allocated sample sizes:
duration_bucket
<=60       145
61-180      14
181-600    138
>600       103
Name: count, dtype: int64

Final bucket counts:
duration_bucket
<=60       145
61-180      14
181-600    138
>600       103
Name: count, dtype: int64
Total rows: 400


E:\temp\ipykernel_9032\367177762.py:24: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  s1.groupby("duration_bucket", group_keys=False)
E:\temp\ipykernel_9032\367177762.py:25: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=min(len(x), alloc1.loc[x.name]), random_state=42))
E:\temp\ipykernel_9032\367177762.py:60: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=Tr

In [12]:
import pandas as pd

df = pd.read_csv("Integrated Data Final (1).csv")
df = df[df["duration_seconds"] != 0].copy()

bins = [0, 60, 180, 600, float("inf")]
labels = ["<=60", "61-180", "181-600", ">600"]
df["duration_bucket"] = pd.cut(df["duration_seconds"], bins=bins, labels=labels)

max_possible = len(df)  
print("Max possible rows:", max_possible)
target_n = 1500

counts = df["duration_bucket"].value_counts().sort_index()
weights = counts / counts.sum()
alloc = (weights * target_n).round().astype(int)

diff = target_n - alloc.sum()
if diff != 0:
    order = (weights * target_n - (weights * target_n).round()).abs().sort_values().index
    alloc.loc[order[:abs(diff)]] += 1 if diff > 0 else -1

alloc = alloc.clip(upper=counts)

df_strat = (
    df.groupby("duration_bucket", group_keys=False)
      .apply(lambda x: x.sample(n=alloc.loc[x.name], random_state=42))
      .reset_index(drop=True)
)

print("Original bucket counts:")
print(counts)
print("\nAllocated sample sizes:")
print(alloc)
print("\nFinal bucket counts:")
print(df_strat["duration_bucket"].value_counts().sort_index())
print("\nTotal rows:", len(df_strat))

df_strat.to_csv(f"Integrated-Data-Final-STRATIFIED_{len(df_strat)}.csv", index=False)


Max possible rows: 1721
Original bucket counts:
duration_bucket
<=60       1110
61-180      103
181-600     124
>600        384
Name: count, dtype: int64

Allocated sample sizes:
duration_bucket
<=60       967
61-180      90
181-600    108
>600       335
Name: count, dtype: int64

Final bucket counts:
duration_bucket
<=60       967
61-180      90
181-600    108
>600       335
Name: count, dtype: int64

Total rows: 1500


E:\temp\ipykernel_9032\3681820237.py:26: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  df.groupby("duration_bucket", group_keys=False)
E:\temp\ipykernel_9032\3681820237.py:27: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda x: x.sample(n=alloc.loc[x.name], random_state=42))
